# From exposed facilities to continuity of care

**Time:** 45 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Connect coastal exposure to a service network; compute a transparent capacity scenario; explain why patient impacts cannot be inferred from counts alone.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/10_continuity_and_equity.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [1]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

Ready: public poster aggregates + clearly labeled fictional facility fixtures.


## Follow the service, not only the building

The poster's next steps call for connecting facility exposure with program indicators. That linkage requires approved data, compatible periods and clear definitions. An annual service count can include repeated visits by the same person. A headcount, a visit count and a treatment-current indicator cannot be substituted for one another.

For this lab, service loads and spare capacities are hypothetical. We model a small network to reason about continuity: a clinic may be dry but unreachable; an alternative site may be reachable but lack staff, medicines or capacity. Decisions need community knowledge about transport costs, stigma, disability access and displacement.

The original study sits in PEPFAR's historical 2022 strategy context. This lesson teaches a planning method, not a statement about current funding, eligibility or clinical care.

In [2]:
# Entirely fictional: annual visit capacity, not patients or real PEPFAR indicators.
network=pd.DataFrame({'site':['Coastal A','Coastal B','Inland C','Inland D'],
    'x':[0,1,3,4],'y':[0,2,1,3],'visit_load':[1000,800,1200,600],
    'spare_capacity':[0,0,500,400],'unavailable':[True,True,False,False]})
edges=[(0,2),(0,3),(1,2),(1,3),(2,3)]
fig=go.Figure()
for a,b in edges:
    fig.add_trace(go.Scatter(x=network.loc[[a,b],'x'],y=network.loc[[a,b],'y'],mode='lines',line=dict(color='#a0b5bb'),showlegend=False,hoverinfo='skip'))
fig.add_trace(go.Scatter(x=network.x,y=network.y,mode='markers+text',text=network.site,textposition='top center',
    marker=dict(size=network.visit_load/40,color=['#cb4260' if x else '#087e8b' for x in network.unavailable]),showlegend=False))
fig.update_layout(title='Fictional referral graph — schematic, not a road map',xaxis=dict(visible=False),yaxis=dict(visible=False))
show(fig)

<iframe title="Interactive teaching figure" style="width:100%;height:560px;border:0" srcdoc="<!doctype html>
<html>
<head>
 <meta charset="utf-8" />
 <style>html, body {height: 100%;}</style>
</head>
<body>
 <div style="height:100%; width:100%;"> <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="../../assets/plotly.min.js"></script> <div id="193d4150-4938-4380-af58-fe981849c8e6" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("193d4150-4938-4380-af58-fe981849c8e6")) { Plotly.newPlot( "193d4150-4938-4380-af58-fe981849c8e6", [{"hoverinfo":"skip","line":{"color":"#a0b5bb"},"mode":"lines","showlegend":false,"x":{"dtype":"i1","bdata":"AAM="},"y":{"dtype":"i1","bdata":"AAE="},"type":"scatter"},{"hoverinfo":"skip","line":{"color":"#a0b5bb"},"mode":"lines","showlegend":false,"x":{"dtype":"i1","bdata":"AAQ="},"y":{"dtype":"i1","bdata":"AAM="},"type":"scatter"},{"hoverinfo":"skip","line":{"color":"#a0b5bb"},"mode":"lines","showlegend":false,"x":{"dtype":"i1","bdata":"AQM="},"y":{"dtype":"i1","bdata":"AgE="},"type":"scatter"},{"hoverinfo":"skip","line":{"color":"#a0b5bb"},"mode":"lines","showlegend":false,"x":{"dtype":"i1","bdata":"AQQ="},"y":{"dtype":"i1","bdata":"AgM="},"type":"scatter"},{"hoverinfo":"skip","line":{"color":"#a0b5bb"},"mode":"lines","showlegend":false,"x":{"dtype":"i1","bdata":"AwQ="},"y":{"dtype":"i1","bdata":"AQM="},"type":"scatter"},{"marker":{"color":["#cb4260","#cb4260","#087e8b","#087e8b"],"size":{"dtype":"f8","bdata":"AAAAAAAAOUAAAAAAAAA0QAAAAAAAAD5AAAAAAAAALkA="}},"mode":"markers+text","showlegend":false,"text":["Coastal A","Coastal B","Inland C","Inland D"],"textposition":"top center","x":{"dtype":"i1","bdata":"AAEDBA=="},"y":{"dtype":"i1","bdata":"AAIBAw=="},"type":"scatter"}], {"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"choropleth":[{"type":"choropleth","colorbar":{"outlinewidth":0,"ticks":""}}],"histogram2d":[{"type":"histogram2d","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"heatmap":[{"type":"heatmap","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"contourcarpet":[{"type":"contourcarpet","colorbar":{"outlinewidth":0,"ticks":""}}],"contour":[{"type":"contour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"surface":[{"type":"surface","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"mesh3d":[{"type":"mesh3d","colorbar":{"outlinewidth":0,"ticks":""}}],"scatter":[{"fillp

In [3]:
disrupted=network.loc[network.unavailable,'visit_load'].sum()
spare=network.loc[~network.unavailable,'spare_capacity'].sum()
reroutable=min(disrupted,spare)
print({'hypothetical_disrupted_annual_visits':int(disrupted),'capacity_only_upper_bound_rerouted':int(reroutable),
       'remaining_visit_capacity_gap':int(disrupted-reroutable)})
# Capacity-only bound assumes all alternative routes and services are usable.
assert disrupted-reroutable == 900

{'hypothetical_disrupted_annual_visits': 1800, 'capacity_only_upper_bound_rerouted': 900, 'remaining_visit_capacity_gap': 900}


In [4]:
alternatives=pd.DataFrame({'option':['Backup power','Access-route improvement','Mobile service','Referral capacity'],
    'illustrative_cost_units':[2,5,3,4],'illustrative_visits_restored':[200,500,350,500]})
alternatives['visits_per_cost_unit']=alternatives.illustrative_visits_restored/alternatives.illustrative_cost_units
display(alternatives)

,option,illustrative_cost_units,illustrative_visits_restored,visits_per_cost_unit
0,Backup power,2,200,100.000000
1,Access-route improvement,5,500,100.000000
2,Mobile service,3,350,116.666667
3,Referral capacity,4,500,125.000000


## Lab: challenge the ranking

Add an equity criterion and a time-to-implement criterion to the options table. Explain why maximizing visits per cost alone could disadvantage remote communities. Avoid inventing precise weights and presenting the outcome as an objective optimum.

**Worked discussion:** The 900-visit gap is a capacity accounting exercise under assumptions, not an estimate of people missing treatment. The graph omits travel times and its edges may fail together. The option impacts may overlap, so adding their benefits could double-count restored services.

**Deliverable:** A scenario memo with an exposure statement, an access constraint, two feasible responses, a stakeholder consultation plan and a monitoring trigger. Keep risk assessment, value judgments and resource choices visible.

Reference: [PEPFAR's 2022 strategic direction](https://www.state.gov/wp-content/uploads/2022/09/PEPFAR-Strategic-Direction_FINAL.pdf); [UNAIDS explanation of cascade denominators](https://www.unaids.org/en/resources/documents/2024/progress-towards-95-95-95).